# 05 bridge build

Question: does the Gradle build of `bridge/` produce the two jars that `okf/bridge.md` specifies,
`golem-host.jar` with the host class and its native methods and the reobfuscated bridge mod, and
does the mod behave on a stock Forge 47.4.18 as the contract says: off with one warning without the
host jar, off with one warning with the host jar but without natives, and on, with every native
called and every action run, when natives answer.

The notebook builds a copy of `bridge/` with Gradle 8.14.3 on Temurin 17, reads both jars, checks
the host class against the Java block of `okf/bridge.md` and the mod against its metadata and the
SRG names of Minecraft, and then boots copies of the Forge server that `tests/fetch.py` installs,
through the java launcher as `run.sh` does, three times. The third boot stands natives in for
Golem: a small C library loaded as a JVMTI agent, whose symbols HotSpot searches for a native
method that no library of its class loader holds, and a minimal offline client of Minecraft 1.20.1
that joins, chats, runs commands, dies, respawns and leaves.

In [ ]:
# Papermill overrides this cell.
# The tree under test, the root of the repository.
golem_root = "."

# Gradle, a path relative to the root or a name on PATH. tests/fetch.py installs it here.
gradle = "tests/.cache/gradle-8.14.3/bin/gradle"
# The JDK 17 that runs Gradle, compiles the bridge and runs the server.
java_home = "tests/.cache/jdk-17.0.21+9"
# The Forge server that tests/fetch.py installs. The notebook boots copies of it alone.
forge_server = "tests/.cache/forge-server"
# The C compiler of the stand in natives.
cc = "cc"
# The port of the copies of the server, which bind 127.0.0.1 alone.
server_port = 25633
# True passes --offline to Gradle, for a machine whose Gradle cache is warm.
gradle_offline = False
# The seconds a boot may take to reach Done, and a stop to end.
boot_timeout = 300

# The pins.
gradle_version = "8.14.3"
java_version = "17.0.21"
class_major = 61
forge_version = "47.4.18"
minecraft_version = "1.20.1"
pack_format = 15

## Setup

The cell finds the repository and the three downloads and prints the versions of Python, Java,
Gradle and the C compiler. The version of Golem comes from `[workspace.package]` of `Cargo.toml`,
which the build of the bridge takes as well.

In [ ]:
import hashlib
import json
import os
import re
import shutil
import socket
import struct
import subprocess
import sys
import tempfile
import threading
import time
import tomllib
import uuid
import zipfile
import zlib
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "tests" / "harness" / "__init__.py").is_file():
        sys.path.insert(0, str(_base / "tests"))
        break

from harness import QUIET, paths, run

P = paths(golem_root)


def located(value):
    # A value with a slash is a path relative to the root, a bare name is looked up on PATH.
    if "/" in value:
        return (P.root / value).resolve()
    found = shutil.which(value)
    assert found, "%s is not on PATH" % value
    return Path(found)


GRADLE = located(gradle)
JAVA_HOME = located(java_home)
JAVA = JAVA_HOME / "bin" / "java"
JAVAP = JAVA_HOME / "bin" / "javap"
FORGE = located(forge_server)
CC = located(cc)
SCRATCH = Path(tempfile.mkdtemp(prefix="golem-05-"))
ENVIRONMENT = dict(os.environ, **QUIET, JAVA_HOME=str(JAVA_HOME))
GOLEM_VERSION = tomllib.loads((P.root / "Cargo.toml").read_text())["workspace"]["package"]["version"]

for program in (GRADLE, JAVA, FORGE):
    assert program.exists(), "%s is missing, python3 -I tests/fetch.py installs it" % program.name

java = subprocess.run([JAVA, "-version"], capture_output=True, text=True, env=ENVIRONMENT)
java_line = next(line for line in java.stderr.splitlines() if "version" in line)
gradle_line = next(line for line in run([GRADLE, "--version"], timeout=300, check=True).stdout.splitlines()
                   if line.startswith("Gradle "))
print("python:", sys.version.split()[0])
print("java:", java_line)
print("gradle:", gradle_line)
print("cc:", run([CC, "--version"], check=True).stdout.splitlines()[0])
print("golem:", GOLEM_VERSION)
assert '"%s"' % java_version in java_line, "the JDK is not Temurin %s: %s" % (java_version, java_line)
assert gradle_line == "Gradle " + gradle_version, "the build is pinned to Gradle %s" % gradle_version

## The build

The cell copies `bridge/` and `Cargo.toml` into a temporary directory, so the build writes nothing
into the tree, and runs `gradle build` there with the JDK passed on the command line, as
`okf/bridge.md` describes. The Gradle cache of the user stays shared, so only the first build on a
machine downloads Minecraft and remaps it.

In [ ]:
TREE = SCRATCH / "tree"
shutil.copytree(P.root / "bridge", TREE / "bridge", ignore=shutil.ignore_patterns("build", ".gradle"))
shutil.copy2(P.root / "Cargo.toml", TREE / "Cargo.toml")
command = [GRADLE, "-p", TREE / "bridge", "--console=plain",
           "-Porg.gradle.java.installations.paths=%s" % JAVA_HOME, "build"]
if gradle_offline:
    command.append("--offline")
started = time.monotonic()
build = subprocess.run([str(part) for part in command], capture_output=True, text=True, env=ENVIRONMENT,
                       timeout=1700)
seconds = time.monotonic() - started
tasks = [line for line in build.stdout.splitlines() if line.startswith("> Task ")]
print("gradle build: exit %d after %.0f s, %d tasks" % (build.returncode, seconds, len(tasks)))
print("\n".join(line for line in tasks if ":mod:" in line))
assert build.returncode == 0, "the build failed:\n" + build.stdout[-6000:] + build.stderr[-6000:]

HOST_JAR = TREE / "bridge" / "host" / "build" / "libs" / "golem-host.jar"
MOD_JAR = TREE / "bridge" / "mod" / "build" / "libs" / ("golem-bridge-%s.jar" % GOLEM_VERSION)
for jar in (HOST_JAR, MOD_JAR):
    assert jar.is_file(), "the build wrote no %s" % jar.relative_to(TREE)
    print("built:", jar.relative_to(TREE), jar.stat().st_size, "bytes")
assert sorted(path.name for path in MOD_JAR.parent.iterdir()) == [MOD_JAR.name], "more than one mod jar"
assert ":mod:reobfJar" in "\n".join(tasks), "ForgeGradle ran no reobfJar"

## Reading class files

The checks read the class files themselves: the version, the methods with their flags and
descriptors, and every member that a class refers to in its constant pool. `javap` cross checks the
descriptors of the host class below.

In [ ]:
ACC_PUBLIC, ACC_STATIC, ACC_NATIVE = 0x0001, 0x0008, 0x0100
REFERENCES = {9: "field", 10: "method", 11: "interface method"}


def read_class(data):
    # Answers the version, the name, the methods and the references of one class file.
    assert data[:4] == b"\xca\xfe\xba\xbe", "no class file"
    minor, major, count = struct.unpack(">HHH", data[4:10])
    pool, offset, index = {}, 10, 1
    while index < count:
        tag = data[offset]
        if tag == 1:
            (length,) = struct.unpack(">H", data[offset + 1:offset + 3])
            pool[index] = ("utf8", data[offset + 3:offset + 3 + length].decode("utf-8", "replace"))
            offset += 3 + length
        elif tag in (3, 4):
            offset += 5
        elif tag in (5, 6):
            offset += 9
            index += 1
        elif tag in (7, 8, 16, 19, 20):
            pool[index] = (tag, struct.unpack(">H", data[offset + 1:offset + 3])[0])
            offset += 3
        elif tag in (9, 10, 11, 12, 17, 18):
            pool[index] = (tag, struct.unpack(">HH", data[offset + 1:offset + 5]))
            offset += 5
        elif tag == 15:
            offset += 4
        else:
            raise AssertionError("constant pool tag %d" % tag)
        index += 1

    def utf8(slot):
        return pool[slot][1]

    access, this = struct.unpack(">HH", data[offset:offset + 4])
    offset += 6
    (interfaces,) = struct.unpack(">H", data[offset:offset + 2])
    offset += 2 + 2 * interfaces
    members = []
    for _ in range(2):
        (number,) = struct.unpack(">H", data[offset:offset + 2])
        offset += 2
        found = []
        for _ in range(number):
            flags, name, descriptor, attributes = struct.unpack(">HHHH", data[offset:offset + 8])
            offset += 8
            for _ in range(attributes):
                (length,) = struct.unpack(">I", data[offset + 2:offset + 6])
                offset += 6 + length
            found.append((flags, utf8(name), utf8(descriptor)))
        members.append(found)
    references = []
    for tag, value in pool.values():
        if tag in REFERENCES:
            owner, pair = value
            name, descriptor = pool[pair][1]
            references.append((REFERENCES[tag], utf8(pool[owner][1]), utf8(name), utf8(descriptor)))
    names = {value for kind, value in pool.values() if kind == "utf8"}
    return {"major": major, "minor": minor, "name": utf8(pool[this][1]), "access": access,
            "fields": members[0], "methods": members[1], "references": references, "strings": names}


def read_manifest(text):
    # Answers the main attributes of a manifest, with its continuation lines joined.
    joined = re.sub(r"\r?\n ", "", text)
    return dict(line.split(": ", 1) for line in joined.splitlines() if ": " in line)


print("reader ready")

## The contract

The Java block of `okf/bridge.md` names every method of the host class. The cell reads it and
turns every signature into the descriptor that the JVM and `RegisterNatives` use.

In [ ]:
TYPES = {"void": "V", "boolean": "Z", "long": "J", "int": "I",
         "String": "Ljava/lang/String;", "String[]": "[Ljava/lang/String;"}
contract = (P.root / "okf" / "bridge.md").read_text()
block = re.search(r"```java\n(.*?)```", contract, re.S).group(1)
assert "package com.skytekx.golem.host;" in block and "public final class Host" in block


def descriptor(result, parameters):
    types = [part.split()[0] for part in parameters.split(",") if part.strip()]
    return "(%s)%s" % ("".join(TYPES[kind] for kind in types), TYPES[result])


NATIVES = {name: descriptor(result, parameters) for result, name, parameters in
           re.findall(r"public static native ([\w\[\]]+) (\w+)\(([^)]*)\);", block, re.S)}
PLAIN = {name: descriptor(result, parameters) for result, name, parameters in
         re.findall(r"public static (?!native)([\w\[\]]+) (\w+)\(([^)]*)\);", block, re.S)}
for name, signature in sorted(NATIVES.items()):
    print("native %-15s %s" % (name, signature))
print("plain  %s" % PLAIN)
assert len(NATIVES) == 14, "okf/bridge.md names %d natives, not 14" % len(NATIVES)
assert PLAIN == {"available": "()Z"}, PLAIN

## The host jar

`golem-host.jar` holds exactly one class, `com/skytekx/golem/host/Host.class`, for Java 17 (class
file version 61), and depends on nothing. Its manifest names the automatic module
`com.skytekx.golem.host`, the version of Golem and the license. Its public methods are the natives
of the contract with their exact descriptors and the plain `available()`, all static.

In [ ]:
with zipfile.ZipFile(HOST_JAR) as jar:
    entries = [name for name in jar.namelist() if not name.endswith("/")]
    host_manifest = read_manifest(jar.read("META-INF/MANIFEST.MF").decode())
    classes = [name for name in entries if name.endswith(".class")]
    host = read_class(jar.read("com/skytekx/golem/host/Host.class"))
print("entries:", entries)
print("manifest:", {key: host_manifest[key] for key in sorted(host_manifest) if key != "Manifest-Version"})
assert classes == ["com/skytekx/golem/host/Host.class"], "the host jar holds %s" % classes
assert sorted(entries) == ["META-INF/MANIFEST.MF", "com/skytekx/golem/host/Host.class"], entries
assert host_manifest.get("Automatic-Module-Name") == "com.skytekx.golem.host", host_manifest
assert host_manifest.get("Implementation-Version") == GOLEM_VERSION, host_manifest
assert host_manifest.get("Bundle-License") == "AGPL-3.0-only", host_manifest
assert (host["major"], host["minor"]) == (class_major, 0), "class file %d.%d" % (host["major"], host["minor"])
assert host["name"] == "com/skytekx/golem/host/Host"

public = {name: (flags, signature) for flags, name, signature in host["methods"] if flags & ACC_PUBLIC}
natives = {name: signature for name, (flags, signature) in public.items() if flags & ACC_NATIVE}
for name, (flags, signature) in sorted(public.items()):
    assert flags & ACC_STATIC, "%s is not static" % name
assert natives == NATIVES, "natives differ from okf/bridge.md: %s" % (set(natives.items()) ^ set(NATIVES.items()))
assert {name: signature for name, (flags, signature) in public.items() if not flags & ACC_NATIVE} == PLAIN
outside = {reference[1] for reference in host["references"]} - {"com/skytekx/golem/host/Host"}
assert all(owner.startswith("java/lang/") for owner in outside), "the host class uses %s" % outside
print("natives: %d of %d with their descriptors, plain: %s, uses only %s" % (
    len(natives), len(NATIVES), sorted(set(public) - set(natives)), sorted(outside)))

javap = run([JAVAP, "-s", "-p", "-cp", HOST_JAR, "com.skytekx.golem.host.Host"], check=True).stdout
seen = re.findall(r"public static native \S+ (\w+)\(.*\);\n\s+descriptor: (\S+)", javap)
assert dict(seen) == NATIVES, "javap reads other descriptors: %s" % seen
print("javap agrees on %d descriptors" % len(seen))

## The mod jar

The mod jar is `golem-bridge-<version>.jar`. Its `mods.toml` names the mod `golem`, the license,
the display test of a server side mod and the version of the manifest, and `pack.mcmeta` holds
pack format 15. No class of the host package and no `.license` file of the tree enters the jar.

In [ ]:
with zipfile.ZipFile(MOD_JAR) as jar:
    entries = [name for name in jar.namelist() if not name.endswith("/")]
    mod_manifest = read_manifest(jar.read("META-INF/MANIFEST.MF").decode())
    mods_toml = tomllib.loads(jar.read("META-INF/mods.toml").decode())
    pack = json.loads(jar.read("pack.mcmeta"))
    mod_classes = {name: read_class(jar.read(name)) for name in entries if name.endswith(".class")}
print("entries:", len(entries), "classes:", sorted(name.rsplit("/", 1)[1] for name in mod_classes))
entry = mods_toml["mods"][0]
print("mods.toml:", {key: mods_toml[key] for key in ("modLoader", "loaderVersion", "license")},
      {key: entry[key] for key in ("modId", "displayName", "version", "displayTest")})
print("pack.mcmeta:", pack)
assert mods_toml["modLoader"] == "javafml" and mods_toml["license"] == "AGPL-3.0-only", mods_toml
assert len(mods_toml["mods"]) == 1 and entry["modId"] == "golem", mods_toml["mods"]
assert entry["displayName"] == "Golem bridge", entry
assert entry["version"] == "${file.jarVersion}", entry
assert entry["displayTest"] == "IGNORE_SERVER_VERSION", entry
dependencies = {item["modId"]: item for item in mods_toml["dependencies"]["golem"]}
assert set(dependencies) == {"forge", "minecraft"} and all(item["mandatory"] for item in dependencies.values())
assert pack == {"pack": {"description": "Golem bridge", "pack_format": pack_format}}, pack
assert mod_manifest.get("Implementation-Version") == GOLEM_VERSION, mod_manifest
assert mod_manifest.get("Implementation-Title") == "Golem bridge", mod_manifest
assert mod_manifest.get("Bundle-License") == "AGPL-3.0-only", mod_manifest
assert not [name for name in entries if name.startswith("com/skytekx/golem/host/")], "the host class is shaded"
assert not [name for name in entries if name.endswith(".license")], "a .license file entered the jar"
assert all(name.startswith("com/skytekx/golem/bridge/") for name in mod_classes), sorted(mod_classes)
assert {(item["major"], item["minor"]) for item in mod_classes.values()} == {(class_major, 0)}
mod_class = [item["name"] for item in mod_classes.values()
             if "Lnet/minecraftforge/fml/common/Mod;" in item["strings"] and "golem" in item["strings"]]
assert mod_class == ["com/skytekx/golem/bridge/GolemBridge"], mod_class
print("the class of @Mod(\"golem\"):", mod_class[0])

## The calls of the mod

The mod listens for every event of the table in `okf/bridge.md`, with the classes of Forge 47, and
calls every native of the contract but `version()`, which `available()` calls, with the descriptors
of the host jar. Every member of Minecraft that it uses carries its SRG name, `m_<n>_` or `f_<n>_`,
since the server runs Minecraft in those names. Only the methods of `java.lang.Object` keep their
names. None of the official names of Mojang that the sources use may appear on a class of
Minecraft.

In [ ]:
EVENTS = [
    "net/minecraftforge/event/server/ServerStartingEvent",
    "net/minecraftforge/event/server/ServerStartedEvent",
    "net/minecraftforge/event/server/ServerStoppingEvent",
    "net/minecraftforge/event/server/ServerStoppedEvent",
    "net/minecraftforge/event/TickEvent$ServerTickEvent",
    "net/minecraftforge/event/entity/player/PlayerEvent$PlayerLoggedInEvent",
    "net/minecraftforge/event/entity/player/PlayerEvent$PlayerLoggedOutEvent",
    "net/minecraftforge/event/ServerChatEvent",
    "net/minecraftforge/event/entity/living/LivingDeathEvent",
    "net/minecraftforge/event/entity/player/PlayerEvent$PlayerRespawnEvent",
    "net/minecraftforge/event/RegisterCommandsEvent",
]
#: The official names of the members of Minecraft that the sources of the mod call.
OFFICIAL = ["getCommands", "performPrefixedCommand", "createCommandSourceStack", "getPlayerList",
            "getPlayer", "sendSystemMessage", "broadcastSystemMessage", "literal", "argument",
            "getStringUUID", "getGameProfile", "level", "dimension", "location", "typeHolder",
            "unwrapKey", "getMsgId", "getServerVersion", "getEntity", "sendSuccess", "sendFailure"]
SRG = re.compile(r"^[mf]_\d+_$")

references = [reference for item in mod_classes.values() for reference in item["references"]]
listened = set()
for item in mod_classes.values():
    for flags, name, signature in item["methods"]:
        listened.update(re.findall(r"\(L([\w/$]+);\)V", signature))
missing = [event for event in EVENTS if event not in listened]
print("events listened for: %d of %d" % (len(EVENTS) - len(missing), len(EVENTS)))
assert not missing, "the mod listens for none of %s" % missing

host_calls = {(name, signature) for kind, owner, name, signature in references
              if owner == "com/skytekx/golem/host/Host"}
expected = {(name, signature) for name, signature in NATIVES.items() if name != "version"}
expected.add(("available", "()Z"))
print("calls of the host class: %d" % len(host_calls))
assert host_calls == expected, "the calls differ from the contract: %s" % (host_calls ^ expected)

minecraft = sorted({reference for reference in references if reference[1].startswith("net/minecraft/")})
kept = {"<init>", "toString", "equals", "hashCode"}
wrong = [reference for reference in minecraft if not SRG.match(reference[2]) and reference[2] not in kept]
official = [reference for reference in minecraft if reference[2] in OFFICIAL]
for kind, owner, name, signature in minecraft:
    print("  %-16s %s.%s" % (kind, owner.rsplit("/", 1)[1], name))
print("members of Minecraft: %d, in SRG names: %d" % (len(minecraft), sum(bool(SRG.match(r[2])) for r in minecraft)))
assert len(minecraft) >= 20, "the mod calls only %d members of Minecraft" % len(minecraft)
assert not wrong, "members of Minecraft without SRG names: %s" % wrong
assert not official, "official names of Mojang in the jar: %s" % official

## A copy of the server

The cell copies the installed server into the temporary directory without its world, logs, crash
reports and mods, so each run starts from a fresh world, and sets the port and the address of the
copy. The installed server stays untouched. The mod is the only jar in `mods/` of the copy. A
boot runs the java launcher of the JDK with the argument files of the server, as `run.sh` does,
waits for `Done`, runs what the case asks on the console and stops the server, and answers what it
printed and logged.

In [ ]:
SERVER = SCRATCH / "server"
ARGS = "libraries/net/minecraftforge/forge/1.20.1-%s/unix_args.txt" % forge_version
TOP = {"world", "logs", "crash-reports", "mods"}
shutil.copytree(FORGE, SERVER, symlinks=True,
                ignore=lambda directory, names: [name for name in names if Path(directory) == FORGE
                                                 and (name in TOP or name.endswith(".log"))])
(SERVER / "mods").mkdir()
shutil.copy2(MOD_JAR, SERVER / "mods" / MOD_JAR.name)
settings = [line for line in (SERVER / "server.properties").read_text().splitlines()
            if not line.startswith(("server-port=", "server-ip=", "online-mode="))]
settings += ["server-port=%d" % server_port, "server-ip=127.0.0.1", "online-mode=false"]
(SERVER / "server.properties").write_text("\n".join(settings) + "\n")
assert (SERVER / ARGS).is_file() and (SERVER / "user_jvm_args.txt").is_file()
print("server copy:", sorted(path.name for path in SERVER.iterdir()))


class Boot:
    # One run of the server copy, with its output read on a thread as it comes.

    def __init__(self, argfile, agent=None):
        for directory in ("logs", "crash-reports"):
            shutil.rmtree(SERVER / directory, ignore_errors=True)
        command = [str(JAVA)] + (["-agentpath:%s" % agent] if agent else [])
        command += ["@user_jvm_args.txt", "@" + argfile, "nogui"]
        self.lines = []
        self.process = subprocess.Popen(command, cwd=SERVER, stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                                        stderr=subprocess.STDOUT, text=True, env=ENVIRONMENT)
        self.reader = threading.Thread(target=self.pump, daemon=True)
        self.reader.start()
        self.started = time.monotonic()
        self.expect("]: Done (", timeout=boot_timeout)
        self.done = time.monotonic() - self.started

    def pump(self):
        for line in self.process.stdout:
            self.lines.append(line.rstrip("\n"))

    def tail(self, count=60):
        return "\n".join(self.lines[-count:]).replace(str(SCRATCH), "<scratch>")

    def expect(self, text, count=1, timeout=60):
        # Waits until `count` lines of the output hold `text`.
        deadline = time.monotonic() + timeout
        while sum(text in line for line in list(self.lines)) < count:
            if self.process.poll() is not None or time.monotonic() > deadline:
                self.process.kill()
                raise AssertionError("no %r after %d s:\n%s" % (text, timeout, self.tail()))
            time.sleep(0.2)

    def console(self, line):
        self.process.stdin.write(line + "\n")
        self.process.stdin.flush()

    def finish(self, stop=True):
        # Stops the server with stop on its console, or waits for a stop it asked itself.
        if stop:
            self.console("stop")
        try:
            self.code = self.process.wait(boot_timeout)
        except subprocess.TimeoutExpired:
            self.process.kill()
            raise AssertionError("the server did not stop:\n" + self.tail())
        self.reader.join(10)
        self.latest = (SERVER / "logs" / "latest.log").read_text(errors="replace")
        self.debug = (SERVER / "logs" / "debug.log").read_text(errors="replace")
        crashes = SERVER / "crash-reports"
        self.crashes = sorted(path.name for path in crashes.iterdir()) if crashes.is_dir() else []
        return self

    def golem(self, level):
        return [line.split("]: ", 1)[1] for line in self.latest.splitlines()
                if "/%s] [golem/]" % level in line]


def check_clean(boot):
    # The checks every boot passes: Done, a clean stop, no fatal line, no crash and no trace of the mod.
    fatal = [line for line in boot.latest.splitlines() if "/FATAL]" in line]
    traces = [line for line in boot.debug.splitlines() if line.strip().startswith("at com.skytekx.golem")]
    print("exit %d, Done after %.1f s, FATAL lines %d, crash reports %d" % (
        boot.code, boot.done, len(fatal), len(boot.crashes)))
    assert boot.code == 0, "the server exited %d:\n%s" % (boot.code, boot.tail())
    assert not fatal, fatal
    assert not boot.crashes, boot.crashes
    for text in ("Stopping the server", "All dimensions are saved"):
        assert text in boot.latest, "the log lacks the stop line %r" % text
    assert "Found valid mod file %s with {golem} mods - versions {%s}" % (MOD_JAR.name, GOLEM_VERSION) in boot.debug
    assert "Missing metadata in pack mod:golem" not in boot.latest, "Forge misses the pack metadata"
    return traces


print("boot ready")

## A stock Forge without the host jar

The server runs with its own `unix_args.txt`, so the host class is missing. The mod must log one
warning that it is off and nothing else, and the server must reach `Done` and stop as without the
mod.

In [ ]:
stock = Boot(ARGS).finish()
traces = check_clean(stock)
warnings, errors = stock.golem("WARN"), stock.golem("ERROR")
print("warnings of the mod:", warnings)
assert warnings == ["The host class of Golem is missing, so the Golem bridge is off"
                    " and the server runs as a stock Forge server"], warnings
assert not errors and not stock.golem("INFO"), (errors, stock.golem("INFO"))
assert "com.skytekx" not in stock.latest, "the log names a class of the bridge"
assert not traces, "a stack trace of the mod:\n" + "\n".join(traces)

## A stock Forge with the host jar

The cell copies `unix_args.txt` of the server and adds `golem-host.jar` to its module path, as the
JVM host of Golem does. The host class is there, but no native is registered, so `available()`
answers false and the mod must log one warning that the natives are missing.

In [ ]:
shutil.copy2(HOST_JAR, SERVER / "golem-host.jar")
original = (SERVER / ARGS).read_text()
changed = [line + ":golem-host.jar" if line.startswith("-p ") else line for line in original.splitlines()]
(SERVER / "golem_unix_args.txt").write_text("\n".join(changed) + "\n")
assert original == (FORGE / ARGS).read_text(), "the copy of unix_args.txt changed"
assert sum(line.startswith("-p ") and line.endswith(":golem-host.jar") for line in changed) == 1

hosted = Boot("golem_unix_args.txt").finish()
traces = check_clean(hosted)
warnings, errors = hosted.golem("WARN"), hosted.golem("ERROR")
print("warnings of the mod:", warnings)
assert warnings == ["The natives of Golem are not registered, so the Golem bridge is off"
                    " and the server runs as a stock Forge server"], warnings
assert not errors and not hosted.golem("INFO"), (errors, hosted.golem("INFO"))
assert "com.skytekx" not in hosted.latest, "the log names a class of the bridge"
assert not traces, "a stack trace of the mod:\n" + "\n".join(traces)

## Natives that stand in for Golem

The C library below implements every native of the host class. It appends each call with the
name of its Java thread to `golem-stub.trace` in the server copy, answers two command names, one
of which Brigadier cannot parse, and queues actions on the console command `golemstub queue`: one
of every kind, an unknown kind, a command without its field and a message to a broken UUID. The
command `golemstub throw` makes the native throw, and `golemstub stop` queues the stop path of
Golem, `save-all flush` and `stop`. A chat message that starts with `cancel` is cancelled.

In [ ]:
STUB_C = r"""
#include <jni.h>
#include <pthread.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#define US "\x1f"
#define QUEUE_MAX 64

static pthread_mutex_t lock = PTHREAD_MUTEX_INITIALIZER;
static char *queue[QUEUE_MAX];
static int queued;
static long ticks;

JNIEXPORT jint JNICALL Agent_OnLoad(JavaVM *vm, char *options, void *reserved)
{
	return JNI_OK;
}

static void push(const char *action)
{
	pthread_mutex_lock(&lock);
	if (queued < QUEUE_MAX)
		queue[queued++] = strdup(action);
	pthread_mutex_unlock(&lock);
}

/* Copies a Java string into out, or the word null for a null reference. */
static void text(JNIEnv *env, jstring value, char *out, size_t size)
{
	const char *chars;

	if (!value) {
		snprintf(out, size, "null");
		return;
	}
	chars = (*env)->GetStringUTFChars(env, value, NULL);
	snprintf(out, size, "%s", chars);
	(*env)->ReleaseStringUTFChars(env, value, chars);
}

static void thread_name(JNIEnv *env, char *out, size_t size)
{
	jclass thread = (*env)->FindClass(env, "java/lang/Thread");
	jmethodID current = (*env)->GetStaticMethodID(env, thread, "currentThread",
						      "()Ljava/lang/Thread;");
	jmethodID get_name = (*env)->GetMethodID(env, thread, "getName", "()Ljava/lang/String;");
	jobject self = (*env)->CallStaticObjectMethod(env, thread, current);
	jstring name = (*env)->CallObjectMethod(env, self, get_name);

	text(env, name, out, size);
	(*env)->DeleteLocalRef(env, name);
	(*env)->DeleteLocalRef(env, self);
	(*env)->DeleteLocalRef(env, thread);
}

static void trace(JNIEnv *env, const char *method, const char *a, const char *b,
		  const char *c, const char *d)
{
	const char *parts[] = { a, b, c, d };
	char name[128];
	FILE *file;

	thread_name(env, name, sizeof(name));
	pthread_mutex_lock(&lock);
	file = fopen("golem-stub.trace", "a");
	if (file) {
		fprintf(file, "%s\t%s", name, method);
		for (int i = 0; i < 4; i++)
			if (parts[i])
				fprintf(file, "\t%s", parts[i]);
		fprintf(file, "\n");
		fclose(file);
	}
	pthread_mutex_unlock(&lock);
}

static jobjectArray strings(JNIEnv *env, char **values, int count)
{
	jclass string = (*env)->FindClass(env, "java/lang/String");
	jobjectArray array = (*env)->NewObjectArray(env, count, string, NULL);

	for (int i = 0; i < count; i++) {
		jstring value = (*env)->NewStringUTF(env, values[i]);

		(*env)->SetObjectArrayElement(env, array, i, value);
		(*env)->DeleteLocalRef(env, value);
	}
	return array;
}

static void player(JNIEnv *env, const char *method, jstring uuid, jstring name, jstring extra)
{
	char a[64], b[64], c[256];

	text(env, uuid, a, sizeof(a));
	text(env, name, b, sizeof(b));
	text(env, extra, c, sizeof(c));
	trace(env, method, a, b, extra ? c : NULL, NULL);
}

#define NATIVE(result, name) JNIEXPORT result JNICALL Java_com_skytekx_golem_host_Host_##name

NATIVE(jstring, version)(JNIEnv *env, jclass host)
{
	trace(env, "version", NULL, NULL, NULL, NULL);
	return (*env)->NewStringUTF(env, "0.0.0-stub");
}

NATIVE(void, serverStarting)(JNIEnv *env, jclass host, jstring forge, jstring minecraft)
{
	char a[64], b[64];

	text(env, forge, a, sizeof(a));
	text(env, minecraft, b, sizeof(b));
	trace(env, "serverStarting", a, b, NULL, NULL);
}

NATIVE(void, serverStarted)(JNIEnv *env, jclass host)
{
	trace(env, "serverStarted", NULL, NULL, NULL, NULL);
	push("broadcast" US "stub: the server started");
}

NATIVE(void, serverStopping)(JNIEnv *env, jclass host)
{
	trace(env, "serverStopping", NULL, NULL, NULL, NULL);
}

NATIVE(void, serverStopped)(JNIEnv *env, jclass host)
{
	char count[32];

	snprintf(count, sizeof(count), "%ld", ticks);
	trace(env, "serverStopped", count, NULL, NULL, NULL);
}

NATIVE(void, tick)(JNIEnv *env, jclass host, jlong nanos)
{
	char value[32];

	if (++ticks > 3)
		return;
	snprintf(value, sizeof(value), "%lld", (long long)nanos);
	trace(env, "tick", value, NULL, NULL, NULL);
}

NATIVE(void, playerJoined)(JNIEnv *env, jclass host, jstring uuid, jstring name)
{
	player(env, "playerJoined", uuid, name, NULL);
}

NATIVE(void, playerLeft)(JNIEnv *env, jclass host, jstring uuid, jstring name)
{
	player(env, "playerLeft", uuid, name, NULL);
}

NATIVE(jboolean, chat)(JNIEnv *env, jclass host, jstring uuid, jstring name, jstring message)
{
	char line[256];

	player(env, "chat", uuid, name, message);
	text(env, message, line, sizeof(line));
	return strncmp(line, "cancel", 6) == 0 ? JNI_TRUE : JNI_FALSE;
}

NATIVE(void, death)(JNIEnv *env, jclass host, jstring uuid, jstring name, jstring cause)
{
	player(env, "death", uuid, name, cause);
}

NATIVE(void, respawn)(JNIEnv *env, jclass host, jstring uuid, jstring name, jstring dimension)
{
	player(env, "respawn", uuid, name, dimension);
}

NATIVE(jobjectArray, commandNames)(JNIEnv *env, jclass host)
{
	char *names[] = { "golemstub", "golem stub" };

	trace(env, "commandNames", NULL, NULL, NULL, NULL);
	return strings(env, names, 2);
}

static void steer(const char *word, const char *uuid)
{
	char action[256];

	if (strcmp(word, "queue") == 0) {
		push("command" US "say stub: a command of golem");
		push("broadcast" US "stub: a broadcast of golem");
		if (strcmp(uuid, "null") != 0) {
			snprintf(action, sizeof(action), "message" US "%s" US "stub: a message", uuid);
			push(action);
		}
		push("teleport" US "somewhere");
		push("command");
		push("message" US "not-a-uuid" US "text");
	} else if (strcmp(word, "stop") == 0) {
		push("command" US "save-all flush");
		push("command" US "stop");
	}
}

NATIVE(jobjectArray, command)(JNIEnv *env, jclass host, jstring name, jstring uuid,
			      jstring sender, jstring arguments)
{
	char a[64], b[64], c[64], d[256], line[512];
	char *lines[] = { line, "stub: a second line" };

	text(env, name, a, sizeof(a));
	text(env, uuid, b, sizeof(b));
	text(env, sender, c, sizeof(c));
	text(env, arguments, d, sizeof(d));
	trace(env, "command", a, b, c, d);
	if (strcmp(d, "throw") == 0) {
		jclass error = (*env)->FindClass(env, "java/lang/IllegalStateException");

		(*env)->ThrowNew(env, error, "stub: the native throws on purpose");
		return NULL;
	}
	steer(d, b);
	snprintf(line, sizeof(line), "stub: %s ran for %s with [%s]", a, c, d);
	return strings(env, lines, 2);
}

/* Traces a drain only when it answers actions, since it runs on every tick. */
NATIVE(jobjectArray, drain)(JNIEnv *env, jclass host)
{
	jobjectArray array;
	char count[32];
	int answered;

	pthread_mutex_lock(&lock);
	array = strings(env, queue, queued);
	for (int i = 0; i < queued; i++)
		free(queue[i]);
	answered = queued;
	queued = 0;
	pthread_mutex_unlock(&lock);
	if (answered) {
		snprintf(count, sizeof(count), "%d", answered);
		trace(env, "drain", count, NULL, NULL, NULL);
	}
	return array;
}
"""

STUB = SCRATCH / "stub"
STUB.mkdir()
(STUB / "stub.c").write_text(STUB_C)
# The JDK keeps jni_md.h in a directory named after the system: linux, or darwin on macOS.
SYSTEM_INCLUDE = JAVA_HOME / "include" / ("darwin" if sys.platform == "darwin" else "linux")
assert (SYSTEM_INCLUDE / "jni_md.h").is_file(), "the JDK holds no %s" % (SYSTEM_INCLUDE / "jni_md.h")
compiled = run([CC, "-shared", "-fPIC", "-O1", "-Wall", "-Werror", "-Wno-unused-parameter",
                "-I%s" % (JAVA_HOME / "include"), "-I%s" % SYSTEM_INCLUDE,
                "-o", STUB / "libgolemstub.so", STUB / "stub.c", "-lpthread"])
assert compiled.returncode == 0, compiled.stdout + compiled.stderr
symbols = re.findall(rb"Java_com_skytekx_golem_host_Host_(\w+)", (STUB / "libgolemstub.so").read_bytes())
print("stand in natives:", sorted({symbol.decode() for symbol in symbols}))
assert {symbol.decode() for symbol in symbols} == set(NATIVES), "the stub misses a native"

## A client of Minecraft

`tests/harness/minecraft.py` holds a client that speaks the protocol of Minecraft 1.20.1, number
763, in offline mode: the handshake, the login with the plugin requests of Forge answered as a
vanilla client would, the keep alive, chat and commands without signatures, the respawn and the
close. It records every system chat message it receives as raw text.

In [ ]:
from harness.minecraft import PROTOCOL, Client, offline_uuid

print("client ready, protocol", PROTOCOL)

## The bridge on

The server runs with the host jar and the stand in natives. The cell drives one session: a
player joins, chats twice, runs the command of the plugin twice, dies, respawns and leaves, then
the console runs the command, reloads the data, runs it again, makes the native throw and asks for
the stop path of Golem, which stops the server through the actions alone.

In [ ]:
TRACE = SERVER / "golem-stub.trace"
TRACE.unlink(missing_ok=True)


def traced(text, timeout=60):
    deadline = time.monotonic() + timeout
    while not (TRACE.is_file() and text in TRACE.read_text()):
        assert time.monotonic() < deadline, "the trace holds no %r" % text
        time.sleep(0.2)


on = Boot("golem_unix_args.txt", agent=STUB / "libgolemstub.so")
on.expect("stub: the server started")
player = Client(server_port, "Tester")
player.login()
on.expect("Tester joined the game")
player.chat("hello golem")
on.expect("<Tester> hello golem")
player.chat("cancel this one")
traced("cancel this one")
player.command("golemstub queue")
on.expect("stub: a broadcast of golem")
player.command("golemstub with some words")
traced("with some words")
on.console("kill Tester")
on.expect("Killed Tester")
player.respawn()
traced("\trespawn\t")
time.sleep(1)
player.close()
on.expect("Tester left the game")
on.console("golemstub queue")
on.expect("stub: a broadcast of golem", count=2)
on.console("reload")
traced("Server thread\tcommandNames")
on.console("golemstub after reload")
on.expect("with [after reload]")
on.console("golemstub throw")
on.expect("The command golemstub failed")
on.console("golemstub stop")
on.finish(stop=False)
traces = check_clean(on)
calls = [line.split("\t") for line in TRACE.read_text().splitlines()]
print("native calls traced: %d" % len(calls))
for call in calls:
    print("  " + " | ".join(call))

## What the natives saw

Every native of the contract was called. `version()` ran once, from the constructor of the mod on
a thread of the mod loader. `commandNames()` ran on the main thread before `serverStarting`, since
Forge builds the commands before it starts the server, and again on the server thread on the
reload. Chat reached Golem on a worker thread of Forge, every other event on the server thread,
and the events came in the order of the life of the server.

In [ ]:
methods = [call[1] for call in calls]
threads = {}
for call in calls:
    threads.setdefault(call[1], set()).add(call[0])
TESTER = offline_uuid("Tester")
print("threads:", {method: sorted(names) for method, names in sorted(threads.items())})
assert set(methods) == set(NATIVES), "natives never called: %s" % (set(NATIVES) - set(methods))
assert methods.count("version") == 1, methods.count("version")
assert all(name.startswith("modloading-worker-") for name in threads["version"]), threads["version"]
assert [call[0] for call in calls if call[1] == "commandNames"] == ["main", "Server thread"], calls
assert methods.index("commandNames") < methods.index("serverStarting")
assert all(name.startswith("Worker-Main-") for name in threads["chat"]), threads["chat"]
elsewhere = {method for method, names in threads.items() if names != {"Server thread"}}
assert elsewhere == {"version", "commandNames", "chat"}, elsewhere
order = ["serverStarting", "serverStarted", "tick", "playerJoined", "chat", "death", "respawn",
         "playerLeft", "serverStopping", "serverStopped"]
assert [methods.index(method) for method in order] == sorted(methods.index(method) for method in order)
assert methods[-2:] == ["serverStopping", "serverStopped"], methods[-2:]

assert ["Server thread", "serverStarting", forge_version, minecraft_version] in calls
assert all(int(call[2]) > 0 for call in calls if call[1] == "tick")
assert int(calls[-1][2]) > 0, "no tick before the stop"
assert player.uuid == TESTER, (player.uuid, TESTER)
assert ["Server thread", "playerJoined", TESTER, "Tester"] in calls
assert ["Server thread", "playerLeft", TESTER, "Tester"] in calls
assert [call[2:] for call in calls if call[1] == "chat"] == [
    [TESTER, "Tester", "hello golem"], [TESTER, "Tester", "cancel this one"]]
assert ["Server thread", "death", TESTER, "Tester", "minecraft:generic_kill"] in calls
assert ["Server thread", "respawn", TESTER, "Tester", "minecraft:overworld"] in calls
assert [call[2:] for call in calls if call[1] == "command"] == [
    ["golemstub", TESTER, "Tester", "queue"], ["golemstub", TESTER, "Tester", "with some words"],
    ["golemstub", "null", "null", "queue"], ["golemstub", "null", "null", "after reload"],
    ["golemstub", "null", "null", "throw"], ["golemstub", "null", "null", "stop"]]
print("every native called: %d of %d, the player %s" % (len(set(methods)), len(NATIVES), TESTER))

## What the game did

The chat message that the natives cancelled never reached the server log. The answers of the
command reached the player and the console. Every action ran on the server thread: the command as
the console, the broadcast to the player and the console, the message to the player alone. The
unknown kind and the command without its field were logged once each although queued twice, the
broken UUID failed once with its stack trace and once more without a line, the throwing native left
the console a failure message, and the name with a space was skipped at the start and on the
reload. The stop path of Golem stopped the server after `save-all flush`.

In [ ]:
received = "\n".join(player.messages)
for text in ("stub: golemstub ran for Tester with [queue]", "stub: golemstub ran for Tester with [with some words]",
             "stub: a second line", "stub: a broadcast of golem", "stub: a message"):
    assert text in received, "the player never saw %r:\n%s" % (text, received)
assert "<Tester> hello golem" in on.latest and "cancel this one" not in on.latest, "the cancel failed"
assert on.latest.count("[Server] stub: a command of golem") == 2, "the command did not run as the console"
assert on.latest.count("]: stub: a broadcast of golem") == 2
assert "stub: golemstub ran for null with [after reload]" in on.latest
assert on.latest.index("Saved the game") < on.latest.index("Stopping the server"), "no save-all flush first"

warnings, errors = on.golem("WARN"), on.golem("ERROR")
print("warnings of the mod:", warnings)
print("errors of the mod:", errors)
skipped = "Golem names the command 'golem stub', which Brigadier cannot parse, so the bridge skips it"
assert warnings == [skipped,
                    "Golem queued an action teleport of a kind the bridge does not know, which the bridge skips",
                    "Golem queued an action command with 0 fields instead of 1, which the bridge skips",
                    skipped], warnings
assert errors == ["The action message failed, and the game goes on",
                  "Host.command failed, and the game goes on"], errors
assert on.golem("INFO") == ["The Golem bridge %s is on" % GOLEM_VERSION], on.golem("INFO")
assert "The command golemstub failed, and the log of the server says why" in on.latest
thrown = [line for line in traces if "PluginCommands" in line or "Actions.message" in line]
assert len(traces) > 0 and thrown, "the guard logged no stack trace"
print("traces of the mod in the log: %d lines, from the two errors above" % len(traces))

## Verdict

The cell clears the temporary directory and prints the counts. A run that reaches it passed.

In [ ]:
shutil.rmtree(SCRATCH)
srg = sum(bool(SRG.match(reference[2])) for reference in minecraft)
print("verdict: the bridge builds golem-host.jar with %d natives and the reobfuscated mod, whose %d members of "
      "Minecraft are %d SRG names and %d methods of Object, stays off with one warning on a stock Forge %s with "
      "and without the host jar, and with stand in natives calls all %d natives and runs every action" % (
          len(NATIVES), len(minecraft), srg, len(minecraft) - srg, forge_version, len(set(methods))))